In [ ]:
# [claude-science] cd to bundle root so inputs/ resolve
import os
os.chdir(os.path.join(os.getcwd(), "../../.."))
try:
    os.makedirs("handoff", exist_ok=True)
except OSError:
    pass

In [2]:
try:  # errored at line 1 in original session — wrapped so replay continues
    import time, numpy as np, bc_model as bc
    
    rng = np.random.default_rng(0)
    x0 = rng.random(200)
    for eps in (0.05, 0.15, 0.30):
        t0 = time.perf_counter()
        xf, st, _ = bc.hk_run(x0, eps)
        nc, neff, lf, cen, sz = bc.cluster_summary(xf)
        t1 = time.perf_counter()
        xd, rd, _ = bc.deffuant_run(x0, eps, rng=np.random.default_rng(1))
        ncd, neffd, lfd, _, _ = bc.cluster_summary(xd)
        print(f"eps={eps}: HK steps={st} nc={nc} neff={neff:.2f} lf={lf:.2f} "
              f"centers={np.round(cen,3)} | t={t1-t0:.2f}s || Deffuant rounds={rd} nc={ncd} neff={neffd:.2f} lf={lfd:.2f} t={time.perf_counter()-t1:.2f}s")
    
except Exception as _e:
    print(f"[replay: cell errored as expected: {type(_e).__name__}: {_e}]")

Traceback (most recent call last):
  File "<kernel:2>", line 1, in <module>
    import time, numpy as np, bc_model as bc
ModuleNotFoundError: No module named 'bc_model'


In [3]:
import os, sys, time, numpy as np
print(os.getcwd(), os.path.exists('bc_model.py'))
sys.path.insert(0, os.getcwd())
import bc_model as bc

rng = np.random.default_rng(0)
x0 = rng.random(200)
for eps in (0.05, 0.15, 0.30):
    t0 = time.perf_counter()
    xf, st, _ = bc.hk_run(x0, eps)
    nc, neff, lf, cen, sz = bc.cluster_summary(xf)
    t1 = time.perf_counter()
    xd, rd, _ = bc.deffuant_run(x0, eps, rng=np.random.default_rng(1))
    ncd, neffd, lfd, _, _ = bc.cluster_summary(xd)
    print(f"eps={eps}: HK steps={st} nc={nc} neff={neff:.2f} lf={lf:.2f} centers={np.round(cen,3)} t={t1-t0:.2f}s "
          f"|| Deff rounds={rd} nc={ncd} neff={neffd:.2f} lf={lfd:.2f} t={time.perf_counter()-t1:.2f}s")


C:\Users\xx\.claude-science\orgs\3b81a10e-d597-46e9-a8b8-12d8d1dd3947\workspaces\f9fa236d-78b7-49af-98cc-0c6c654630af True
eps=0.05: HK steps=8 nc=8 neff=7.29 lf=0.19 centers=[0.052 0.135 0.274 0.395 0.572 0.705 0.837 0.932] t=0.00s || Deff rounds=600 nc=10 neff=7.42 lf=0.18 t=0.01s
eps=0.15: HK steps=8 nc=3 neff=2.89 lf=0.40 centers=[0.202 0.487 0.841] t=0.00s || Deff rounds=300 nc=3 neff=2.90 lf=0.39 t=0.00s
eps=0.3: HK steps=6 nc=1 neff=1.00 lf=1.00 centers=[0.55] t=0.00s || Deff rounds=100 nc=2 neff=1.03 lf=0.98 t=0.00s


In [6]:
import importlib, pandas as pd, numpy as np, time
importlib.reload(bc)

N, SEEDS = 400, 30
EPS = np.round(np.arange(0.01, 0.505, 0.01), 3)
rows = []
t0 = time.perf_counter()
for eps in EPS:
    for s in range(SEEDS):
        x0 = np.random.default_rng(1000 + s).random(N)
        xf, st, _ = bc.hk_run(x0, eps)
        nc, neff, lf, _, _ = bc.cluster_summary(xf)
        rows.append(dict(model="Hegselmann-Krause", eps=eps, seed=s, N=N,
                         n_clusters=nc, n_eff=neff, largest_frac=lf, steps=st))
        xd, rd, _ = bc.deffuant_run(x0, eps, mu=0.5, rounds=12000,
                                    rng=np.random.default_rng(5000 + s))
        ncd, neffd, lfd, _, _ = bc.cluster_summary(xd)
        rows.append(dict(model="Deffuant", eps=eps, seed=s, N=N,
                         n_clusters=ncd, n_eff=neffd, largest_frac=lfd, steps=rd))
sweep = pd.DataFrame(rows)
sweep.to_csv("sweep_eps.csv", index=False)
agg = (sweep.groupby(["model", "eps"])
       .agg(n_eff=("n_eff", "mean"), n_eff_sd=("n_eff", "std"),
            largest_frac=("largest_frac", "mean"),
            p_consensus=("largest_frac", lambda v: (v > 0.95).mean()),
            steps=("steps", "mean")).reset_index())
agg.to_csv("sweep_eps_summary.csv", index=False)
print(f"{len(sweep)} runs in {time.perf_counter()-t0:.1f}s")
for m in agg.model.unique():
    a = agg[agg.model == m]
    thr = a[a.p_consensus >= 0.95].eps.min()
    print(m, "| consensus(>=95% of runs) from eps =", thr,
          "| n_eff at eps=0.05,0.10,0.20:",
          [round(float(a.loc[np.isclose(a.eps, e), "n_eff"].iloc[0]), 2) for e in (0.05, 0.10, 0.20)])


3000 runs in 59.9s
Deffuant | consensus(>=95% of runs) from eps = 0.31 | n_eff at eps=0.05,0.10,0.20: [8.4, 4.28, 2.02]
Hegselmann-Krause | consensus(>=95% of runs) from eps = 0.26 | n_eff at eps=0.05,0.10,0.20: [6.84, 3.47, 1.94]


In [7]:
import networkx as nx
# --- (a) topology sweep: HK on four graphs with identical mean degree
Nn, K, SEEDS_T = 200, 8, 12
EPS_T = np.round(np.arange(0.025, 0.501, 0.025), 4)

def make_graph(kind, N, k, seed):
    if kind == "complete":
        return None
    if kind == "Erdos-Renyi":
        return nx.gnm_random_graph(N, N * k // 2, seed=seed)
    if kind == "Watts-Strogatz":
        return nx.watts_strogatz_graph(N, k, 0.1, seed=seed)
    if kind == "Barabasi-Albert":
        return nx.barabasi_albert_graph(N, k // 2, seed=seed)

rows = []
t0 = time.perf_counter()
for kind in ["complete", "Erdos-Renyi", "Watts-Strogatz", "Barabasi-Albert"]:
    for s in range(SEEDS_T):
        G = make_graph(kind, Nn, K, 200 + s)
        A = None if G is None else bc.adjacency_with_self_loops(G, Nn)
        x0 = np.random.default_rng(1000 + s).random(Nn)
        for eps in EPS_T:
            xf, st, _ = bc.hk_run(x0, eps, A=A, max_steps=600)
            nc, neff, lf, _, _ = bc.cluster_summary(xf)
            rows.append(dict(topology=kind, eps=eps, seed=s, N=Nn, mean_degree=K,
                             n_clusters=nc, n_eff=neff, largest_frac=lf, steps=st))
topo = pd.DataFrame(rows); topo.to_csv("sweep_topology.csv", index=False)
print("topology sweep", len(topo), f"{time.perf_counter()-t0:.0f}s")

# --- (b) finite-size behaviour of the consensus transition (HK, complete graph)
rows = []
t0 = time.perf_counter()
for Nf, seeds in [(50, 40), (100, 40), (200, 40), (400, 30), (800, 20)]:
    for eps in np.round(np.arange(0.12, 0.351, 0.01), 3):
        for s in range(seeds):
            x0 = np.random.default_rng(7000 + s).random(Nf)
            xf, st, _ = bc.hk_run(x0, eps)
            nc, neff, lf, _, _ = bc.cluster_summary(xf)
            rows.append(dict(N=Nf, eps=eps, seed=s, n_clusters=nc, n_eff=neff,
                             largest_frac=lf, steps=st))
fs = pd.DataFrame(rows); fs.to_csv("sweep_finite_size.csv", index=False)
print("finite-size sweep", len(fs), f"{time.perf_counter()-t0:.0f}s")
print(fs.assign(cons=fs.largest_frac > 0.95).groupby("N")
      .apply(lambda d: d.groupby("eps").cons.mean().pipe(lambda s: s[s >= 0.95].index.min()),
             include_groups=False))


topology sweep 960 16s
finite-size sweep 4080 28s
N
50     0.32
100    0.29
200    0.26
400    0.26
800    0.25
dtype: float64


In [8]:
print([n for n in dir() if 'figure' in n or 'panel' in n or 'style' in n])
# scaling coefficient a in  n_eff ~ a/eps  over the fragmented regime
for m in ["Hegselmann-Krause", "Deffuant"]:
    d = sweep[(sweep.model == m) & (sweep.eps <= 0.15) & (sweep.eps >= 0.02)]
    g = d.groupby("eps").n_eff.mean()
    a = float(np.sum((1 / g.index.values) * g.values) / np.sum((1 / g.index.values) ** 2))
    resid = g.values - a / g.index.values
    print(m, f"a={a:.3f}  (1/a={1/a:.2f})  max|resid|={np.abs(resid).max():.2f}")
print(agg[agg.model=="Hegselmann-Krause"].set_index("eps").steps.loc[[0.05,0.15,0.2,0.26,0.4]].round(1).to_dict())
print(topo.groupby("topology").apply(lambda d: d.groupby("eps").largest_frac.mean().pipe(lambda s: s[s>0.95].index.min()), include_groups=False).to_dict())
print(topo[np.isclose(topo.eps,0.3)].groupby("topology")[["n_eff","largest_frac","steps"]].mean().round(2).to_string())


['apply_figure_style', 'figure_style__META_GREY', 'figure_style__apply_figure_style', 'figure_style__bar_with_points', 'figure_style__end_of_line_labels', 'figure_style__focal_palette', 'figure_style__goodness_arrow', 'figure_style__panel_crops', 'figure_style__panel_letter', 'figure_style__set_frame', 'figure_style__strip_with_median', 'figure_style__two_tier_label', 'panel_crops', 'panel_letter']
Hegselmann-Krause a=0.333  (1/a=3.01)  max|resid|=0.39
Deffuant a=0.412  (1/a=2.43)  max|resid|=0.43
{0.05: 9.5, 0.15: 8.5, 0.2: 7.4, 0.26: 11.1, 0.4: 4.0}
{'Barabasi-Albert': 0.25, 'Erdos-Renyi': 0.225, 'Watts-Strogatz': 0.225, 'complete': 0.275}
                 n_eff  largest_frac   steps
topology                                    
Barabasi-Albert   1.05          0.98   50.25
Erdos-Renyi       1.03          0.99   54.33
Watts-Strogatz    1.04          0.98  281.33
complete          1.00          1.00    5.42


In [9]:
fit = {}
for m in ["Hegselmann-Krause", "Deffuant"]:
    for col in ["n_clusters", "n_eff"]:
        g = sweep[(sweep.model == m) & (sweep.eps.between(0.02, 0.15))].groupby("eps")[col].mean()
        a = float(np.sum((1/g.index.values)*g.values)/np.sum((1/g.index.values)**2))
        fit[(m, col)] = 1/a
print({f"{m[:4]}/{c}": round(v, 2) for (m, c), v in fit.items()})

# trajectories, N=200
EPS_TRAJ = [0.06, 0.15, 0.30]
x0t = np.random.default_rng(12).random(200)
traj = {}
for eps in EPS_TRAJ:
    _, st, T = bc.hk_run(x0t, eps, keep_traj=True)
    traj[("Hegselmann-Krause", eps)] = T
    _, rd, Td = bc.deffuant_run(x0t, eps, mu=0.5, rounds=12000, keep_every=10,
                                rng=np.random.default_rng(3))
    traj[("Deffuant", eps)] = Td
    print(eps, "HK steps", st, "clusters", bc.cluster_summary(T[-1])[0],
          "| Deff rounds", rd, "clusters", bc.cluster_summary(Td[-1])[0],
          "neff", round(bc.cluster_summary(Td[-1])[1], 2))


{'Hegs/n_clusters': 2.58, 'Hegs/n_eff': 3.01, 'Deff/n_clusters': 1.91, 'Deff/n_eff': 2.43}
0.06 HK steps 8 clusters 7 | Deff rounds 1000 clusters 7 neff 5.66
0.15 HK steps 6 clusters 3 | Deff rounds 400 clusters 3 neff 2.96
0.3 HK steps 10 clusters 1 | Deff rounds 300 clusters 2 neff 2.0


In [10]:
help(apply_figure_style); help(panel_letter); help(figure_style__end_of_line_labels); help(figure_style__focal_palette)

Help on function apply_figure_style in module __main__:

apply_figure_style(*, frame='open', font=None, sizes=(8, 7, 6), grid=False)
    Set matplotlib rcParams for publication-grade output. Call once before plotting.
    
    This sets mechanics (role-mapped font-size ladder, outward ticks, frameless
    legends, 300-dpi save, Type-42 embedded fonts) — not a house aesthetic.
    Frame, font and the size ladder are parameters.
    
    frame : 'open' (bottom+left spines, default) | 'boxed' (all four) | 'none'
    font  : sans-serif family name; None = system default sans-serif
    sizes : (base, secondary, tick) — titles/axis-labels, legend/annotation, ticks
    grid  : whether to draw axes.grid (default False)

Help on function panel_letter in module __main__:

panel_letter(ax, letter, dx=-0.18, dy=1.02, case='lower', fontsize=None)
    §5.7: bold panel letter outside top-left of axes. case ∈ {'lower','upper'}.

Help on function end_of_line_labels in module __main__:

end_of_line_labe

In [11]:
import matplotlib as mpl, matplotlib.pyplot as plt
apply_figure_style(frame='open', sizes=(9, 8, 7))

MODELS = ["Hegselmann-Krause", "Deffuant"]
fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.4))
for i, m in enumerate(MODELS):
    for j, eps in enumerate(EPS_TRAJ):
        ax = axes[i, j]
        T = traj[(m, eps)]
        step = 10 if m == "Deffuant" else 1
        t = np.arange(T.shape[0]) * step
        t[0] = 0
        ax.plot(np.where(t == 0, 0.5, t), T, lw=0.5, alpha=0.35, color="#3a4a6b")
        nc, neff, lf, cen, sz = bc.cluster_summary(T[-1])
        ax.set_xscale("log")
        ax.set_xlim(0.5, max(t.max() * 1.6, 20))
        ax.set_ylim(-0.03, 1.03)
        ax.set_yticks([0, 0.5, 1])
        ax.annotate(f"{nc} cluster{'s' if nc > 1 else ''}\nlargest {lf:.0%}",
                    xy=(0.97, 0.5), xycoords="axes fraction", ha="right", va="center",
                    fontsize=8, color="#1b1b1b")
        if i == 0:
            ax.set_title(f"$\\varepsilon$ = {eps:.2f}", pad=6)
        if j == 0:
            ax.set_ylabel(f"{m}\n\nopinion")
            ax.set_xlabel("sweeps (one update per agent)")
        else:
            ax.set_yticklabels([])
        panel_letter(ax, "abcdef"[i * 3 + j], dx=-0.10 if j else -0.26, dy=1.04)
fig.suptitle("One parameter — the confidence threshold $\\varepsilon$ — selects fragmentation,\n"
             "polarization or consensus, in both update rules", y=1.005, fontsize=9)
fig.tight_layout(w_pad=0.6, h_pad=1.4)
fig.savefig("fig1_trajectories.png", dpi=300, bbox_inches="tight")

r = fig.canvas.get_renderer()
texts = [(t_, t_.get_window_extent(r)) for t_ in fig.findobj(mpl.text.Text)
         if t_.get_text().strip() and t_.get_visible()]
print("text overlaps:", sum(1 for i, (a, ba) in enumerate(texts) for b, bb in texts[i+1:] if ba.overlaps(bb)))


text overlaps: 12


In [12]:
for ax in axes.ravel():
    for a_ in ax.texts:
        a_.set_bbox(dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.82))
        a_.set_position((0.97, 0.52))
fig.suptitle("The confidence threshold $\\varepsilon$ sets how many opinion clusters survive:\n"
             "fragmentation, then polarization, then consensus", y=1.005, fontsize=9)
fig.savefig("fig1_trajectories.png", dpi=300, bbox_inches="tight")
host.view_image("fig1_trajectories.png", crop=(0.0, 0.45, 0.5, 1.0))


[view_image] 'fig1_trajectories' [2147×1346px] → crop x=0..1074 y=606..1346 [1074×740px] → '_view_001_fig1_trajectories.png'


In [13]:
plt.close(fig)
fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.4))
for i, m in enumerate(MODELS):
    for j, eps in enumerate(EPS_TRAJ):
        ax = axes[i, j]
        T = traj[(m, eps)]
        step = 10 if m == "Deffuant" else 1
        t = np.arange(T.shape[0]) * step
        ax.plot(np.where(t == 0, 0.5, t), T, lw=0.5, alpha=0.35, color="#3a4a6b")
        nc, neff, lf, cen, sz = bc.cluster_summary(T[-1])
        ax.set_xscale("log"); ax.set_xlim(0.5, max(t.max() * 1.6, 20))
        ax.set_ylim(-0.03, 1.03); ax.set_yticks([0, 0.5, 1])
        ax.text(0.97, 0.52, f"{nc} cluster{'s' if nc > 1 else ''}\nlargest {lf:.0%}",
                transform=ax.transAxes, ha="right", va="center", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.85))
        if i == 0:
            ax.set_title(f"$\\varepsilon$ = {eps:.2f}", pad=6)
        if j == 0:
            ax.set_ylabel(f"{m}\n\nopinion")
            ax.set_xlabel("sweeps (one update per agent)")
        else:
            ax.set_yticklabels([])
        panel_letter(ax, "abcdef"[i * 3 + j], dx=-0.10 if j else -0.26, dy=1.04)
fig.suptitle("The confidence threshold $\\varepsilon$ sets how many opinion clusters survive:\n"
             "fragmentation, then polarization, then consensus", y=1.005, fontsize=9)
fig.tight_layout(w_pad=0.6, h_pad=1.4)
fig.savefig("fig1_trajectories.png", dpi=300, bbox_inches="tight")
host.view_image("fig1_trajectories.png", crop=(0.0, 0.45, 0.45, 1.0))


[view_image] 'fig1_trajectories' [2140×1346px] → crop x=0..963 y=606..1346 [963×740px] → '_view_002_fig1_trajectories.png'


In [14]:
C = {"Hegselmann-Krause": "#1f5fa9", "Deffuant": "#d1702f"}
fig2, ax2 = plt.subplots(1, 3, figsize=(7.4, 2.7))

# --- (a) cluster count vs eps, log-log, with the 1/(2 eps) rule
ax = ax2[0]
for m in MODELS:
    a_ = agg[agg.model == m].sort_values("eps")
    ax.fill_between(a_.eps, a_.n_eff - a_.n_eff_sd, a_.n_eff + a_.n_eff_sd,
                    color=C[m], alpha=0.18, lw=0)
    ax.plot(a_.eps, a_.n_eff, color=C[m], lw=1.4, marker="o", ms=2.6)
e = np.linspace(0.012, 0.52, 200)
ax.plot(e, 1 / (2 * e), color="0.45", lw=1.0, ls="--")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks([0.02, 0.05, 0.1, 0.2, 0.5]); ax.set_xticklabels(["0.02", "0.05", "0.1", "0.2", "0.5"])
ax.set_yticks([1, 2, 5, 10, 20]); ax.set_yticklabels(["1", "2", "5", "10", "20"])
ax.minorticks_off(); ax.set_ylim(0.8, 30)
ax.set_xlabel("confidence threshold $\\varepsilon$")
ax.set_ylabel("surviving clusters (effective)")
ax.set_title("Cluster count follows a $1/\\varepsilon$ law", pad=6)
ax.annotate("Deffuant", xy=(0.021, 17), color=C["Deffuant"], fontsize=8)
ax.annotate("Hegselmann–Krause", xy=(0.036, 7.0), color=C["Hegselmann-Krause"], fontsize=8)
ax.annotate("$1/(2\\varepsilon)$ rule", xy=(0.30, 1.75), color="0.35", fontsize=8)

# --- (b) consensus probability vs eps for increasing N
ax = ax2[1]
fsg = (fs.assign(cons=fs.largest_frac > 0.95).groupby(["N", "eps"]).cons.mean().reset_index())
greys = plt.cm.Greys(np.linspace(0.35, 0.95, fsg.N.nunique()))
for col, (Nv, d) in zip(greys, fsg.groupby("N")):
    ax.plot(d.eps, d.cons, color=col, lw=1.3, marker="o", ms=2.4)
    ax.annotate(f"N={Nv}", xy=(d.eps.iloc[-1] + 0.004, 1.0 - 0.055 * list(fsg.N.unique()).index(Nv)),
                color=col, fontsize=7, va="center")
ax.set_xlim(0.115, 0.40); ax.set_ylim(-0.04, 1.12)
ax.set_xlabel("confidence threshold $\\varepsilon$")
ax.set_ylabel("fraction of runs at consensus")
ax.set_title("Larger crowds agree more easily", pad=6)

# --- (c) time to converge
ax = ax2[2]
for m in MODELS:
    a_ = agg[agg.model == m].sort_values("eps")
    ax.plot(a_.eps, a_.steps, color=C[m], lw=1.4, marker="o", ms=2.6)
    k = a_.steps.idxmax()
    ax.annotate(f"peak at $\\varepsilon$={a_.loc[k, 'eps']:.2f}",
                xy=(a_.loc[k, "eps"], a_.loc[k, "steps"]), xytext=(8, 10 if m == MODELS[0] else -16),
                textcoords="offset points", fontsize=7.5, color=C[m],
                arrowprops=dict(arrowstyle="-", color=C[m], lw=0.7))
ax.set_yscale("log"); ax.set_ylim(3, 3000)
ax.set_yticks([10, 100, 1000]); ax.set_yticklabels(["10", "100", "1k"]); ax.minorticks_off()
ax.set_xlabel("confidence threshold $\\varepsilon$")
ax.set_ylabel("sweeps to convergence")
ax.set_title("Slowest right at the transition", pad=6)

for k, ax in enumerate(ax2):
    panel_letter(ax, "abc"[k], dx=-0.26, dy=1.06)
fig2.tight_layout(w_pad=1.6)
fig2.savefig("fig2_phase_diagram.png", dpi=300, bbox_inches="tight")
r = fig2.canvas.get_renderer()
tx = [(t_, t_.get_window_extent(r)) for t_ in fig2.findobj(mpl.text.Text) if t_.get_text().strip() and t_.get_visible()]
print("overlaps:", [(a.get_text(), b.get_text()) for i, (a, ba) in enumerate(tx) for b, bb in tx[i+1:] if ba.overlaps(bb)])


overlaps: [('confidence threshold $\\varepsilon$', '0.02'), ('confidence threshold $\\varepsilon$', '0.05'), ('confidence threshold $\\varepsilon$', '0.1'), ('confidence threshold $\\varepsilon$', '0.2'), ('confidence threshold $\\varepsilon$', '0.5'), ('surviving clusters (effective)', '10'), ('surviving clusters (effective)', '20'), ('N=50', 'N=100'), ('N=100', 'N=200'), ('N=200', 'N=400'), ('N=400', 'N=800'), ('confidence threshold $\\varepsilon$', '0.2'), ('confidence threshold $\\varepsilon$', '0.3'), ('confidence threshold $\\varepsilon$', '0.0'), ('confidence threshold $\\varepsilon$', '0.2'), ('confidence threshold $\\varepsilon$', '0.4')]


In [15]:
plt.close(fig2)
fig2, ax2 = plt.subplots(1, 3, figsize=(7.6, 2.8))
ax = ax2[0]
for m in MODELS:
    a_ = agg[agg.model == m].sort_values("eps")
    ax.fill_between(a_.eps, a_.n_eff - a_.n_eff_sd, a_.n_eff + a_.n_eff_sd, color=C[m], alpha=0.18, lw=0)
    ax.plot(a_.eps, a_.n_eff, color=C[m], lw=1.4, marker="o", ms=2.6)
ax.plot(e, 1 / (2 * e), color="0.45", lw=1.0, ls="--")
ax.set(xscale="log", yscale="log", ylim=(0.8, 30),
       xlabel="confidence threshold $\\varepsilon$", ylabel="surviving clusters (effective)")
ax.set_xticks([0.02, 0.05, 0.1, 0.2, 0.5]); ax.set_xticklabels(["0.02", "0.05", "0.1", "0.2", "0.5"])
ax.set_yticks([1, 2, 5, 10, 20]); ax.set_yticklabels(["1", "2", "5", "10", "20"]); ax.minorticks_off()
ax.set_title("Cluster count follows a $1/\\varepsilon$ law", pad=6)
ax.annotate("Deffuant", xy=(0.021, 17), color=C["Deffuant"], fontsize=8)
ax.annotate("Hegselmann–Krause", xy=(0.034, 6.6), color=C["Hegselmann-Krause"], fontsize=8)
ax.annotate("$1/(2\\varepsilon)$ rule", xy=(0.285, 1.85), color="0.35", fontsize=8)

ax = ax2[1]
Ns = sorted(fsg.N.unique())
greys = plt.cm.Greys(np.linspace(0.45, 0.95, len(Ns)))
for i, Nv in enumerate(Ns):
    d = fsg[fsg.N == Nv]
    ax.plot(d.eps, d.cons, color=greys[i], lw=1.3, marker="o", ms=2.4)
    ax.text(0.325, 0.70 - 0.095 * i, f"N = {Nv}", color=greys[i], fontsize=7.5, va="center")
ax.set(xlim=(0.115, 0.385), ylim=(-0.04, 1.08), xlabel="confidence threshold $\\varepsilon$",
       ylabel="fraction of runs at consensus")
ax.set_title("Larger crowds agree more easily", pad=6)

ax = ax2[2]
for m in MODELS:
    a_ = agg[agg.model == m].sort_values("eps")
    ax.plot(a_.eps, a_.steps, color=C[m], lw=1.4, marker="o", ms=2.6)
hk = agg[agg.model == "Hegselmann-Krause"].sort_values("eps")
k = hk.steps.idxmax()
ax.annotate(f"peak at $\\varepsilon$ = {hk.loc[k,'eps']:.2f}", xy=(hk.loc[k, "eps"], hk.loc[k, "steps"]),
            xytext=(14, 16), textcoords="offset points", fontsize=7.5, color=C["Hegselmann-Krause"],
            arrowprops=dict(arrowstyle="-", color=C["Hegselmann-Krause"], lw=0.7))
ax.annotate("Deffuant", xy=(0.30, 330), color=C["Deffuant"], fontsize=8)
ax.annotate("Hegselmann–Krause", xy=(0.17, 4.6), color=C["Hegselmann-Krause"], fontsize=8)
ax.set(yscale="log", ylim=(3, 4000), xlabel="confidence threshold $\\varepsilon$",
       ylabel="sweeps to convergence")
ax.set_yticks([10, 100, 1000]); ax.set_yticklabels(["10", "100", "1k"]); ax.minorticks_off()
ax.set_title("Pairwise mixing is ~100$\\times$ slower;\nHK slows near its transition", pad=6, fontsize=8.5)
for i, a_ in enumerate(ax2):
    panel_letter(a_, "abc"[i], dx=-0.27, dy=1.07)
fig2.tight_layout(w_pad=1.8)
fig2.savefig("fig2_phase_diagram.png", dpi=300, bbox_inches="tight")
print("HK max-steps eps:", float(hk.loc[k, "eps"]), "| Deffuant steps at eps 0.05/0.5:",
      [round(float(agg.loc[(agg.model=="Deffuant") & np.isclose(agg.eps, x), "steps"].iloc[0]), 0) for x in (0.05, 0.5)])


HK max-steps eps: 0.23 | Deffuant steps at eps 0.05/0.5: [1817.0, 200.0]


In [16]:
ax2[2].texts[-1].set_position((0.02, 3.55))
fig2.savefig("fig2_phase_diagram.png", dpi=300, bbox_inches="tight")

tg = (topo.assign(cons=topo.largest_frac > 0.95)
      .groupby(["topology", "eps"])
      .agg(n_eff=("n_eff", "mean"), steps=("steps", "mean"), p_cons=("cons", "mean")).reset_index())
thr = {k: float(d[d.p_cons >= 0.95].eps.min()) for k, d in tg.groupby("topology")}
TC = {"complete": "0.45", "Erdos-Renyi": "#1f5fa9", "Watts-Strogatz": "#2d8a5e", "Barabasi-Albert": "#7a4ba0"}
TL = {"complete": "complete graph", "Erdos-Renyi": "Erdős–Rényi",
      "Watts-Strogatz": "Watts–Strogatz", "Barabasi-Albert": "Barabási–Albert"}
order = ["complete", "Erdos-Renyi", "Watts-Strogatz", "Barabasi-Albert"]
print(thr, {k: round(v, 1) for k, v in tg[np.isclose(tg.eps, 0.3)].set_index("topology").steps.items()})

fig3, ax3 = plt.subplots(1, 3, figsize=(7.6, 2.8))
for k in order:
    d = tg[tg.topology == k].sort_values("eps")
    sty = dict(color=TC[k], lw=1.4, marker="o", ms=2.6, ls="--" if k == "complete" else "-")
    ax3[0].plot(d.eps, d.n_eff, **sty)
    ax3[1].plot(d.eps, d.steps, **sty)
ax3[0].set(xscale="log", yscale="log", ylim=(0.8, 30), xlabel="confidence threshold $\\varepsilon$",
           ylabel="surviving clusters (effective)")
ax3[0].set_xticks([0.025, 0.05, 0.1, 0.2, 0.5]); ax3[0].set_xticklabels(["0.025", "0.05", "0.1", "0.2", "0.5"])
ax3[0].set_yticks([1, 2, 5, 10, 20]); ax3[0].set_yticklabels(["1", "2", "5", "10", "20"]); ax3[0].minorticks_off()
ax3[0].set_title("Sparse networks fragment less", pad=6)
for k, xy in zip(order, [(0.028, 5.2), (0.028, 11.5), (0.075, 13.5), (0.20, 3.4)]):
    ax3[0].annotate(TL[k], xy=xy, color=TC[k], fontsize=7.5)
ax3[1].set(yscale="log", xlabel="confidence threshold $\\varepsilon$", ylabel="sweeps to convergence")
ax3[1].set_yticks([10, 100, 1000]); ax3[1].set_yticklabels(["10", "100", "1k"]); ax3[1].minorticks_off()
ax3[1].set_title("Local ties slow agreement by $10$–$100\\times$", pad=6, fontsize=8.5)
for k, xy in zip(order, [(0.33, 4.4), (0.36, 33), (0.30, 420), (0.055, 19)]):
    ax3[1].annotate(TL[k], xy=xy, color=TC[k], fontsize=7.5, ha="right" if k == "Barabasi-Albert" else "left")

y = np.arange(len(order))[::-1]
for i, k in enumerate(order):
    ax3[2].hlines(y[i], 0, thr[k], color=TC[k], lw=0.9, alpha=0.6)
    ax3[2].plot([thr[k]], [y[i]], "o", color=TC[k], ms=6)
    ax3[2].annotate(f"{thr[k]:.3f}", xy=(thr[k] + 0.012, y[i]), va="center", fontsize=7.5, color=TC[k])
ax3[2].set_yticks(y, [TL[k] for k in order], fontsize=7.5)
ax3[2].set(xlim=(0, 0.345), ylim=(-0.6, 3.6),
           xlabel="smallest $\\varepsilon$ with consensus in all runs")
ax3[2].set_title("Mean-field needs the widest\nconfidence to agree", pad=6, fontsize=8.5)
ax3[2].spines["left"].set_visible(False); ax3[2].tick_params(axis="y", length=0)
for i, a_ in enumerate(ax3):
    panel_letter(a_, "abc"[i], dx=-0.27 if i < 2 else -0.55, dy=1.07)
fig3.tight_layout(w_pad=2.0)
fig3.savefig("fig3_topology.png", dpi=300, bbox_inches="tight")


{'Barabasi-Albert': 0.3, 'Erdos-Renyi': 0.225, 'Watts-Strogatz': 0.275, 'complete': 0.275} {'Barabasi-Albert': 50.2, 'Erdos-Renyi': 54.3, 'Watts-Strogatz': 281.3, 'complete': 5.4}


In [17]:
rows = []
EPS_R = np.round(np.arange(0.17, 0.351, 0.01), 3)
t0 = time.perf_counter()
for kind in order:
    for s in range(24):
        G = make_graph(kind, Nn, K, 200 + s)
        A = None if G is None else bc.adjacency_with_self_loops(G, Nn)
        x0 = np.random.default_rng(1000 + s).random(Nn)
        for eps in EPS_R:
            xf, st, _ = bc.hk_run(x0, eps, A=A, max_steps=1500)
            nc, neff, lf, _, _ = bc.cluster_summary(xf)
            rows.append(dict(topology=kind, eps=eps, seed=s, N=Nn, mean_degree=K,
                             n_clusters=nc, n_eff=neff, largest_frac=lf, steps=st))
topo_r = pd.DataFrame(rows); topo_r.to_csv("sweep_topology_transition.csv", index=False)
tr = (topo_r.assign(cons=topo_r.largest_frac > 0.95).groupby(["topology", "eps"])
      .agg(p_cons=("cons", "mean"), n_eff=("n_eff", "mean")).reset_index())
thr = {k: float(d[d.p_cons >= 0.95].eps.min()) for k, d in tr.groupby("topology")}
half = {k: float(d[d.p_cons >= 0.5].eps.min()) for k, d in tr.groupby("topology")}
print(f"{len(topo_r)} runs {time.perf_counter()-t0:.0f}s"); print("all-runs:", thr); print("half-runs:", half)
cross = tg.pivot(index="eps", columns="topology", values="n_eff")
print(cross.round(2).to_string())


1824 runs 24s
all-runs: {'Barabasi-Albert': 0.28, 'Erdos-Renyi': 0.24, 'Watts-Strogatz': 0.29, 'complete': 0.28}
half-runs: {'Barabasi-Albert': 0.22, 'Erdos-Renyi': 0.19, 'Watts-Strogatz': 0.23, 'complete': 0.23}
topology  Barabasi-Albert  Erdos-Renyi  Watts-Strogatz  complete
eps                                                             
0.025               20.67        19.85           21.65     12.75
0.050               16.70        17.93           23.81      6.98
0.075                8.49         6.69           18.02      4.27
0.100                4.69         3.09            9.34      3.68
0.125                2.87         2.25            3.88      2.94
0.150                2.26         1.94            2.43      2.19
0.175                1.46         1.50            1.66      1.94
0.200                1.25         1.16            1.18      1.74
0.225                1.22         1.06            1.11      1.31
0.250                1.08         1.05            1.09      1.16
0.275  

In [18]:
for t_ in ax2[2].texts:
    if t_.get_text() == "c":
        t_.set_position((-0.27, 1.07))
    elif t_.get_text().startswith("Hegselmann"):
        t_.set_position((0.02, 3.6))
fig2.savefig("fig2_phase_diagram.png", dpi=300, bbox_inches="tight")
print([(t_.get_text(), np.round(t_.get_position(), 3)) for t_ in ax2[2].texts])


[('peak at $\\varepsilon$ = 0.23', array([14, 16])), ('Deffuant', array([3.0e-01, 3.3e+02])), ('Hegselmann–Krause', array([0.02, 3.6 ])), ('c', array([-0.27,  1.07]))]


In [19]:
plt.close(fig3)
fig3, ax3 = plt.subplots(1, 3, figsize=(7.8, 2.9))
for k in order:
    d = tg[tg.topology == k].sort_values("eps")
    sty = dict(color=TC[k], lw=1.4, marker="o", ms=2.6, ls="--" if k == "complete" else "-")
    ax3[0].plot(d.eps, d.n_eff, **sty); ax3[1].plot(d.eps, d.steps, **sty)
ax3[0].set(xscale="log", yscale="log", ylim=(0.8, 40), xlabel="confidence threshold $\\varepsilon$",
           ylabel="surviving clusters (effective)")
ax3[0].set_xticks([0.025, 0.05, 0.1, 0.2, 0.5]); ax3[0].set_xticklabels(["0.025", "0.05", "0.1", "0.2", "0.5"])
ax3[0].set_yticks([1, 2, 5, 10, 20]); ax3[0].set_yticklabels(["1", "2", "5", "10", "20"]); ax3[0].minorticks_off()
ax3[0].set_title("Sparse ties fragment more at low $\\varepsilon$,\nless near the transition", pad=6, fontsize=8.5)
for k, xy, ha in zip(order, [(0.028, 3.0), (0.052, 9.0), (0.105, 17.0), (0.062, 3.9)], ["left"] * 4):
    ax3[0].annotate(TL[k], xy=xy, color=TC[k], fontsize=7.5, ha=ha)
ax3[1].set(yscale="log", ylim=(2.5, 1500), xlim=(0.0, 0.53),
           xlabel="confidence threshold $\\varepsilon$", ylabel="sweeps to convergence")
ax3[1].set_yticks([10, 100, 1000]); ax3[1].set_yticklabels(["10", "100", "1k"]); ax3[1].minorticks_off()
ax3[1].set_title("Every sparse topology converges\n10–100$\\times$ slower", pad=6, fontsize=8.5)
for k, xy, ha in zip(order, [(0.52, 3.6), (0.52, 44), (0.52, 290), (0.07, 700)], ["right", "right", "right", "left"]):
    ax3[1].annotate(TL[k], xy=xy, color=TC[k], fontsize=7.5, ha=ha)

y = np.arange(len(order))[::-1]
for i, k in enumerate(order):
    ax3[2].plot([half[k], thr[k]], [y[i]] * 2, color=TC[k], lw=1.6, alpha=0.5, solid_capstyle="round")
    ax3[2].plot([half[k]], [y[i]], "o", color="white", mec=TC[k], mew=1.4, ms=5.5)
    ax3[2].plot([thr[k]], [y[i]], "o", color=TC[k], ms=5.5)
    ax3[2].annotate(f"{half[k]:.2f}–{thr[k]:.2f}", xy=(thr[k] + 0.006, y[i]), va="center",
                    fontsize=7.5, color=TC[k])
ax3[2].set_yticks(y, [TL[k] for k in order], fontsize=7.5)
ax3[2].set(xlim=(0.17, 0.345), ylim=(-0.6, 3.75), xticks=[0.18, 0.22, 0.26, 0.30],
           xlabel="confidence threshold $\\varepsilon$")
ax3[2].set_title("Only random sparse ties lower\nthe consensus threshold", pad=6, fontsize=8.5)
ax3[2].spines["left"].set_visible(False); ax3[2].tick_params(axis="y", length=0)
ax3[2].plot([], [], "o", color="white", mec="0.3", mew=1.4, ms=5.5, label="half of runs")
ax3[2].plot([], [], "o", color="0.3", ms=5.5, label="all runs")
ax3[2].legend(loc="lower left", bbox_to_anchor=(-0.02, -0.02), fontsize=7, handletextpad=0.2,
              labelspacing=0.25, title="consensus in", title_fontsize=7)
for i, a_ in enumerate(ax3):
    panel_letter(a_, "abc"[i], dx=-0.27 if i < 2 else -0.62, dy=1.07)
fig3.tight_layout(w_pad=2.2)
fig3.savefig("fig3_topology.png", dpi=300, bbox_inches="tight")
r = fig3.canvas.get_renderer()
tx = [(t_, t_.get_window_extent(r)) for t_ in fig3.findobj(mpl.text.Text) if t_.get_text().strip() and t_.get_visible()]
print("text-text overlaps:", [(a.get_text()[:14], b.get_text()[:14]) for i, (a, ba) in enumerate(tx) for b, bb in tx[i+1:] if ba.overlaps(bb)])


text-text overlaps: [('confidence thr', '0.025'), ('confidence thr', '0.05'), ('confidence thr', '0.1'), ('confidence thr', '0.2'), ('confidence thr', '0.5'), ('Erdős–Rényi', 'sweeps to conv'), ('Watts–Strogatz', 'sweeps to conv'), ('Barabási–Alber', 'sweeps to conv'), ('confidence thr', '0.0'), ('confidence thr', '0.2'), ('confidence thr', '0.4'), ('Every sparse t', 'c'), ('confidence thr', '0.18'), ('confidence thr', '0.22'), ('confidence thr', '0.26'), ('confidence thr', '0.30')]


In [20]:
for a_ in ax3[:2]:
    for t_ in list(a_.texts):
        if t_.get_text() in TL.values():
            t_.remove()
for k, xy in zip(order, [(0.245, 2.6), (0.245, 5.4), (0.245, 11.5), (0.245, 24)]):
    ax3[0].annotate(TL[k], xy=xy, color=TC[k], fontsize=7.5, ha="left")
for k, xy, ha in zip(order, [(0.012, 3.4), (0.52, 19), (0.52, 450), (0.07, 700)],
                     ["left", "right", "right", "left"]):
    ax3[1].annotate(TL[k], xy=xy, color=TC[k], fontsize=7.5, ha=ha)
ax3[1].set_title("Sparse topologies converge\n10–100$\\times$ slower", pad=6, fontsize=8.5)
ax3[2].get_legend().remove()
ax3[2].set_ylim(-0.6, 4.3)
ax3[2].annotate("half of runs\nat consensus", xy=(half["complete"], 3.08), xytext=(half["complete"] - 0.004, 4.25),
                fontsize=7, color="0.3", ha="center", va="top",
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.6, shrinkB=4))
ax3[2].annotate("all runs", xy=(thr["complete"], 3.08), xytext=(thr["complete"] + 0.004, 4.25),
                fontsize=7, color="0.3", ha="center", va="top",
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.6, shrinkB=4))
panel_letter(ax3[2], "c", dx=-0.45, dy=1.07)
for t_ in ax3[2].texts:
    if t_.get_text() == "c" and t_.get_position()[0] < -0.5:
        t_.remove()
fig3.tight_layout(w_pad=2.2)
fig3.savefig("fig3_topology.png", dpi=300, bbox_inches="tight")


In [22]:
s = pd.read_csv("sweep_eps.csv"); f = pd.read_csv("sweep_finite_size.csv")
t1 = pd.read_csv("sweep_topology.csv"); t2 = pd.read_csv("sweep_topology_transition.csv")
cons = lambda d, q=0.95: (d.assign(c=d.largest_frac > 0.95).groupby("eps").c.mean()
                          .pipe(lambda v: v[v >= q].index.min()))
print("eps* :", {m: round(float(cons(d)), 3) for m, d in s.groupby("model")})
print("finite:", {int(n): round(float(cons(d)), 2) for n, d in f.groupby("N")})
print("Deff steps 0.05/0.5:", [round(float(s[(s.model=="Deffuant") & np.isclose(s.eps,x)].steps.mean())) for x in (0.05,0.5)])
print("HK peak eps:", float(s[s.model=="Hegselmann-Krause"].groupby("eps").steps.mean().idxmax()))
print("neff@0.05:", t1[np.isclose(t1.eps,0.05)].groupby("topology").n_eff.mean().round(1).to_dict())
print("steps@0.30:", t1[np.isclose(t1.eps,0.30)].groupby("topology").steps.mean().round(1).to_dict())
print("topo eps* all/half:", {k: (round(float(cons(d)),2), round(float(cons(d,0.5)),2)) for k,d in t2.groupby("topology")})


eps* : {'Deffuant': 0.31, 'Hegselmann-Krause': 0.26}
finite: {50: 0.32, 100: 0.29, 200: 0.26, 400: 0.26, 800: 0.25}
Deff steps 0.05/0.5: [1817, 200]
HK peak eps: 0.23
neff@0.05: {'Barabasi-Albert': 16.7, 'Erdos-Renyi': 17.9, 'Watts-Strogatz': 23.8, 'complete': 7.0}
steps@0.30: {'Barabasi-Albert': 50.2, 'Erdos-Renyi': 54.3, 'Watts-Strogatz': 281.3, 'complete': 5.4}
topo eps* all/half: {'Barabasi-Albert': (0.28, 0.22), 'Erdos-Renyi': (0.24, 0.19), 'Watts-Strogatz': (0.29, 0.23), 'complete': (0.28, 0.23)}
